# ULAL - Unified Lifelong Action Learning

The principal notebook. It does not train anything; it reads every section's saved result
and assembles the one unified story: continual learning of a video action recogniser under
a moving freeze boundary, and the accuracy/memory trade-off across five orders of magnitude
of replay budget.

Run the study notebooks first (1.0 -> 2.0 -> 4.1..4.6 -> 5.0 -> 6.0 -> 7.0 -> 8.0 -> 9.0).
Each writes a JSON to Drive; this notebook collects them. Sections that have not run are
simply skipped.

| Section | Axis it measures | File |
|---|---|---|
| 1 backbone      | which frozen representation | stage1_choose_backbone.json |
| 2 embeddings    | the ceiling | stage2_embedding_diagnostics.json |
| 4 continual     | forgetting per method | stage1_arm_*.json |
| 5 memory        | accuracy vs bytes | stage5_memory_ablation.json |
| 6 GAN           | 2 KB/class generative replay | stage6_gil.json |
| 7 meta          | few-shot speed | stage7_fewshot.json |
| 8 distillation  | model size | stage8_kd.json |
| 9 adaptation    | YouTube domain shift | stage9_da.json |

## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

## **Download DATASET**

Drive only - every result lives on Drive.

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=False)

In [ ]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

## **Load every section**

Each JSON is optional; a missing one just leaves that axis blank.

In [ ]:
STAGE_FILES = {
    "backbone": "stage1_choose_backbone.json",
    "ceiling":  "stage2_embedding_diagnostics.json",
    "memory":   "stage5_memory_ablation.json",
    "gil":      "stage6_gil.json",
    "fewshot":  "stage7_fewshot.json",
    "kd":       "stage8_kd.json",
    "da":       "stage9_da.json",
}

STAGES = {}
for key, fn in STAGE_FILES.items():
    p = f"{cfg.RESULTS_DIR}/{fn}"
    if os.path.exists(p):
        with open(p) as f:
            STAGES[key] = json.load(f)
    else:
        STAGES[key] = None

ARMS = []
for p in sorted(glob(f"{cfg.RESULTS_DIR}/stage1_arm_*.json")):
    with open(p) as f:
        ARMS.append(json.load(f))

CEILING = STAGES["ceiling"]["ceiling"] if STAGES["ceiling"] else None
print("loaded:", [k for k, v in STAGES.items() if v is not None], f"| CL arms: {len(ARMS)}")

## **The pipeline at a glance**

Each axis in one line, read against the ceiling.

In [ ]:
print("=" * 60)
print("ULAL - unified summary")
print("=" * 60)

if CEILING is not None:
    print(f"CEILING (joint linear probe) : {CEILING:.2%}")

if STAGES["backbone"]:
    b = STAGES["backbone"]
    print(f"BACKBONE                     : {b['best_model']} chosen")

if ARMS:
    best = max(ARMS, key=lambda a: a["metrics"]["AA"])
    print(f"CONTINUAL best method        : {best['arm']}  AA={best['metrics']['AA']:.2%}")

if STAGES["memory"]:
    tr = STAGES["memory"]["trials"]
    top = max(tr, key=lambda t: t["AA"])
    print(f"MEMORY best config           : {top['strategy']} x{top['per_class']}  AA={top['AA']:.2%}  {top['bytes']/1e3:.0f} KB/class")

if STAGES["gil"]:
    g = STAGES["gil"]
    gauss = g.get("gauss_test_acc")
    extra = f" vs gaussian {gauss:.2%}" if gauss is not None else ""
    print(f"GAN generative replay        : gen {g['gen_test_acc']:.2%}{extra} vs real {g['real_test_acc']:.2%}  ({g['bytes_per_class']/1e3:.1f} KB/class)")

if STAGES["fewshot"]:
    fs = STAGES["fewshot"]["trials"]
    hi = max(fs, key=lambda t: t["K"])
    print(f"META few-shot (K={hi['K']})         : reptile {hi['reptile']:.2%} vs standard {hi['standard']:.2%}")

if STAGES["kd"]:
    k = STAGES["kd"]
    print(f"DISTILLATION                 : student {k['student_acc']:.2%} vs teacher {k['teacher_acc']:.2%}  ({k['teacher_params']/k['student_params']:.1f}x smaller)")

if STAGES["da"]:
    d = STAGES["da"]
    print(f"DOMAIN ADAPT (YouTube)       : {d['yt_acc_before']:.2%} -> {d['yt_acc_after']:.2%}  (UCF {d['ucf_acc_before']:.2%} -> {d['ucf_acc_after']:.2%})")
print("=" * 60)

## **Master figure: accuracy vs memory**

Every replay-based method on one log-memory axis: the CL arms, the memory sweep, and the
2 KB/class generator, all under the same ceiling. This is the contribution in one picture.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for a in ARMS:
    x = max(a.get("bytes_per_class", 0), 0.5)
    ax.scatter(x, a["metrics"]["AA"] * 100, s=80, color="tab:blue")
    ax.annotate(a["arm"], (x, a["metrics"]["AA"] * 100), textcoords="offset points",
                xytext=(6, 4), fontsize=8)

if STAGES["memory"]:
    for t in STAGES["memory"]["trials"]:
        if t["bytes"] > 0:
            ax.scatter(t["bytes"], t["AA"] * 100, s=25, color="tab:orange", alpha=0.6)

if STAGES["gil"]:
    g = STAGES["gil"]
    ax.scatter(max(g["bytes_per_class"], 0.5), g["gen_test_acc"] * 100, s=110,
               marker="*", color="tab:green")
    ax.annotate("GAN", (max(g["bytes_per_class"], 0.5), g["gen_test_acc"] * 100),
                textcoords="offset points", xytext=(6, 4), fontsize=9)

if CEILING is not None:
    ax.axhline(CEILING * 100, ls="--", color="gray", label=f"ceiling {CEILING:.0%}")

ax.set_xscale("log")
ax.set_xlabel("memory per class (bytes, log scale)")
ax.set_ylabel("accuracy (%)")
ax.set_title("ULAL - accuracy vs memory across every method")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/ulal_master_figure.png", dpi=150, bbox_inches="tight")
plt.show()

## **Save**

One unified summary JSON for the report.

In [ ]:
summary = {
    "ceiling":       CEILING,
    "backbone":      STAGES["backbone"]["best_model"] if STAGES["backbone"] else None,
    "cl_best":       max(ARMS, key=lambda a: a["metrics"]["AA"])["arm"] if ARMS else None,
    "sections_present": [k for k, v in STAGES.items() if v is not None],
    "n_cl_arms":     len(ARMS),
}
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f"{cfg.RESULTS_DIR}/ulal_unified_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=float)
print(f"Saved -> {cfg.RESULTS_DIR}/ulal_unified_summary.json")